# Export YOLOv8n to TensorRT

This notebook exports YOLOv8n to ONNX and builds `yolov8_trt/1/model.plan` with `trtexec`.

Build this TensorRT plan in the same Triton image and on the same GPU class that will serve it.

In [ ]:
!pip install ultralytics onnx

In [ ]:
from pathlib import Path

from ultralytics import YOLO

repo = Path.cwd()
engine_path = repo / "yolov8_trt" / "1" / "model.plan"
engine_path.parent.mkdir(parents=True, exist_ok=True)

model = YOLO("yolov8n.pt")
onnx_path = Path(model.export(format="onnx", imgsz=640, dynamic=False, simplify=True, opset=17))
target_onnx = repo / "yolov8n.onnx"
if onnx_path.resolve() != target_onnx.resolve():
    target_onnx.write_bytes(onnx_path.read_bytes())

print(f"ONNX: {target_onnx}")
print(f"TensorRT plan target: {engine_path}")

In [ ]:
!trtexec \
  --onnx=yolov8n.onnx \
  --saveEngine=yolov8_trt/1/model.plan \
  --fp16 \
  --shapes=images:1x3x640x640

In [ ]:
from pathlib import Path

plan = Path("yolov8_trt/1/model.plan")
assert plan.exists(), "TensorRT plan was not created"
print(f"Created {plan} ({plan.stat().st_size / (1024 * 1024):.1f} MiB)")